# Gradient boosting — perinatal_anc cohort

Train **gradient boosting** on the `perinatal_anc` cohort and tune it with cluster-grouped
cross-validation on the training rows, then score the held-out test split once (section 4).

Owner: Jeff  ·  **`random_state=42` so results reproduce.**

## 1 — Load the data

In [ ]:
import sys
from pathlib import Path

# make the repo root importable when run from ml/notebooks/modeling
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.src.dataset import build_cohort, cv_folds, SELECTED, OUTCOMES

COHORT = "perinatal_anc"
# load the combined, cleaned cohort
df = build_cohort(COHORT)

# training rows for tuning; test rows are used once, in section 4
train = df[df["split"] == "train"].reset_index(drop=True)
print(f"train rows: {len(train):,}  clusters: {train['cluster_id'].nunique():,}")

## 2 — Features and target
Features are the cohort's `SELECTED` columns minus the outcome and the design columns.

In [ ]:
TARGET = OUTCOMES[COHORT]
DESIGN = ["split", "cluster_id", "country", "weight"]

# features = SELECTED minus outcome and design columns
FEATURES = [c for c in SELECTED[COHORT] if c not in DESIGN + [TARGET]]
X_train, y_train = train[FEATURES], train[TARGET]

# event count and rate (the constant-baseline PR-AUC)
print(f"{len(FEATURES)} features: {FEATURES}")
print(f"events: {int(y_train.sum()):,}  rate: {y_train.mean():.2%}")
# blanks per feature (first pregnancies have no interval by design)
print(X_train.isna().sum()[lambda s: s > 0].to_dict())

## 3 — Train and tune
`HistGradientBoostingClassifier`: supports `class_weight='balanced'` for the rare outcome
(no resampling) and handles blanks natively, so first-pregnancy intervals stay blank.
Tuned with `GridSearchCV` on `cv_folds()` (5 cluster-grouped folds), PR-AUC first.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# balanced class weights for the imbalance; no early stopping so max_iter is tuned
gb = HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False, random_state=42)
# all features are numeric, so the pipeline is just the model
pipe = Pipeline([("model", gb)])

# small grid of simple models: 48 settings x 5 folds
GRID = {
    "model__learning_rate": [0.02, 0.05],
    "model__max_iter": [50, 100],
    "model__max_leaf_nodes": [4, 7],
    "model__min_samples_leaf": [20, 80, 150],
    "model__l2_regularization": [1.0, 5.0],
}

# cluster-grouped folds from dataset.py
folds = cv_folds(train, COHORT)
search = GridSearchCV(pipe, GRID, cv=folds, scoring=["average_precision", "roc_auc"],
                      refit="average_precision", n_jobs=1)
# fit on training rows only
search.fit(X_train, y_train)

In [ ]:
# per-fold scores of the best setting
res, k = search.cv_results_, search.best_index_
fold_scores = pd.DataFrame({
    "pr_auc": [res[f"split{i}_test_average_precision"][k] for i in range(len(folds))],
    "roc_auc": [res[f"split{i}_test_roc_auc"][k] for i in range(len(folds))],
    # constant baseline: PR-AUC of a flat score = event rate in the fold
    "baseline_pr_auc": [y_train.iloc[val].mean() for _, val in folds],
}, index=pd.Index(range(1, len(folds) + 1), name="fold"))

# mean and sd over folds
summary = fold_scores.agg(["mean", "std"]).T.round(4)
print(summary)
print("best params:", {p.replace("model__", ""): v for p, v in search.best_params_.items()})

### CV results (training rows only)

- PR-AUC **0.079 ± 0.027** over 5 cluster-grouped folds, against a constant baseline of 0.031 (about 2.5×).
- ROC-AUC **0.625 ± 0.039**.
- Best setting: learning_rate 0.05, max_iter 50, max_leaf_nodes 4, min_samples_leaf 80, l2_regularization 1.
- A wider grid (up to 300 trees, 15 leaves) scored lower (PR-AUC 0.072), so simpler models suit this cohort.
- Fold scores vary most here (sd 0.027): only 412 events in the training rows.

In [ ]:
# fold scores against the constant baseline
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(fold_scores.index, fold_scores["pr_auc"], label="gradient boosting")
axes[0].plot(fold_scores.index, fold_scores["baseline_pr_auc"], "k--o", label="constant baseline")
axes[0].set(title="CV PR-AUC by fold", xlabel="fold", ylabel="PR-AUC")
axes[0].legend()
axes[1].bar(fold_scores.index, fold_scores["roc_auc"])
axes[1].axhline(0.5, color="k", ls="--")
axes[1].set(title="CV ROC-AUC by fold", xlabel="fold", ylabel="ROC-AUC", ylim=(0.4, 1))
fig.suptitle(f"{COHORT}: gradient boosting, best grid setting")
fig.tight_layout()
plt.show()

## 4 — Evaluate on the test split

The best CV setting, already refit on all training rows by `GridSearchCV`, scores the held-out
test clusters once. Nothing is retuned after this. PR-AUC is the headline; the checklist comparison
flags the same share of test pregnancies as the current high-risk checklist.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

# test rows: the held-out clusters, used once
test = df[df["split"] == "test"].reset_index(drop=True)
X_test, y_test = test[FEATURES], test[TARGET]
print(f"test rows: {len(test):,}  events: {int(y_test.sum()):,}  rate: {y_test.mean():.2%}")

# risk scores from the best setting (refit on all training rows)
model = search.best_estimator_
score = model.predict_proba(X_test)[:, 1]

In [ ]:
# test PR-AUC and ROC-AUC, pooled and per country (country is not a feature)
def metrics(mask):
    y, s = y_test[mask], score[mask]
    return {"rows": int(mask.sum()), "events": int(y.sum()), "baseline_pr_auc": y.mean(),
            "pr_auc": average_precision_score(y, s), "roc_auc": roc_auc_score(y, s)}


test_scores = pd.DataFrame({
    "pooled": metrics(np.ones(len(test), bool)),
    "Kenya": metrics((test["country"] == "Kenya").to_numpy()),
    "Tanzania": metrics((test["country"] == "Tanzania").to_numpy()),
}).T
print(test_scores.round(3))

# CV figures for comparison
print(summary.loc[["pr_auc", "roc_auc"]])

In [ ]:
# checklist flag: same rule as checklist_flag in dataset.py (not in SELECTED, so rebuilt here)
checklist = (
    (test["maternal_age"] < 18) | (test["maternal_age"] >= 35)
    | (test["prior_live_births"] >= 5) | (test["preceding_interval_m"] < 24)
    | (test["multiple"] == 1) | (test["prior_stillbirth"] == 1)
).to_numpy()
n_flag = int(checklist.sum())

# model flags the same number of pregnancies: the highest scores
model_flag = np.zeros(len(test), bool)
model_flag[np.argsort(-score, kind="stable")[:n_flag]] = True

# events caught, sensitivity and PPV at the same flag rate
events = y_test.to_numpy() == 1
compare = pd.DataFrame({
    "flagged": [n_flag, n_flag],
    "events_caught": [int((checklist & events).sum()), int((model_flag & events).sum())],
}, index=["checklist", "gradient boosting"])
compare["sensitivity"] = compare["events_caught"] / events.sum()
compare["ppv"] = compare["events_caught"] / compare["flagged"]
print(f"flag rate: {checklist.mean():.1%} of test pregnancies ({n_flag:,})")
print(compare.round(3))

# many rows share a score at the cut-off: range of events the model could catch
cut = np.sort(score)[::-1][n_flag - 1]
above, tied = score > cut, score == cut
slots = n_flag - int(above.sum())
low = int((above & events).sum()) + max(0, slots - int((tied & ~events).sum()))
high = int((above & events).sum()) + min(slots, int((tied & events).sum()))
print(f"{int(tied.sum())} rows tied at the cut-off; model catches {low}-{high} events")

In [ ]:
# calibration: mean predicted vs observed, and 10 equal-width bins
print(f"mean predicted {score.mean():.3f}  observed rate {y_test.mean():.3f}")
obs, pred = calibration_curve(y_test, score, n_bins=10)

# PR curve with the constant baseline and the checklist point; reliability plot
prec, rec, _ = precision_recall_curve(y_test, score)
pooled_pr = test_scores.loc["pooled", "pr_auc"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(rec, prec, label=f"gradient boosting (PR-AUC {pooled_pr:.3f})")
axes[0].axhline(y_test.mean(), color="k", ls="--", label=f"constant baseline ({y_test.mean():.3f})")
axes[0].plot(compare.loc["checklist", "sensitivity"], compare.loc["checklist", "ppv"], "rs",
             label="checklist")
axes[0].set(title="Test precision-recall", xlabel="recall (sensitivity)", ylabel="precision (PPV)")
axes[0].legend()
axes[1].plot(pred, obs, "o-", label="gradient boosting")
axes[1].plot([0, 1], [0, 1], "k--", label="perfect calibration")
axes[1].set(title="Test reliability (10 bins)", xlabel="mean predicted", ylabel="observed rate")
axes[1].legend()
fig.suptitle(f"{COHORT}: gradient boosting on the test split")
fig.tight_layout()
plt.show()

### Test results

- **PR-AUC 0.079** on 3,310 test pregnancies (108 deaths), against a constant baseline of 0.033: about 2.4× better than flagging at random.
- It held up: test PR-AUC equals the CV mean (0.079 ± 0.027). ROC-AUC 0.622 vs 0.625 ± 0.039 in CV.
- Kenya is stronger than Tanzania: PR-AUC 0.102 vs 0.066 (baselines 0.030 and 0.036), ROC-AUC 0.658 vs 0.569. Tanzania is barely above chance.
- The checklist flags 44% of test pregnancies (1,457) and catches 49 of 108 deaths (sensitivity 0.45, PPV 0.034).
- Flagging the same 1,457 by model score catches 63 deaths (sensitivity 0.58, PPV 0.043): 14 more. 490 rows share the score at the cut-off, so the model catches 60–74 depending on how ties fall; even the low end beats the checklist by 11.
- Probabilities are not risks: mean predicted 0.47 against an observed rate of 0.033, because `class_weight="balanced"` inflates them. The reliability plot still rises with the score, so use the score to rank, not as a probability.
- For the final model: gradient boosting beats the checklist for ANC at the same workload. Pick between it and LR/RF on test PR-AUC once those notebooks are done.

## 5 — Save the result

Values for `db.metrics.save_metrics(...)` (one row in `model_metrics`):

| cohort | algorithm | pr_auc | roc_auc | recall_at_flag | beats_checklist | test_events |
|---|---|---|---|---|---|---|
| perinatal_anc | gradient_boosting | 0.079 | 0.622 | 0.583 | True | 108 |

`recall_at_flag` is the model's sensitivity at the checklist's flag rate (44.0% of test rows).
To save, set `SAVE_TO_DB = True` below once the app Postgres is up; re-running replaces the row (upsert).

In [ ]:
# row for model_metrics, built from the section 4 results
row = {
    "cohort": COHORT,
    "algorithm": "gradient_boosting",
    "pr_auc": float(test_scores.loc["pooled", "pr_auc"]),
    "roc_auc": float(test_scores.loc["pooled", "roc_auc"]),
    "recall_at_flag": float(compare.loc["gradient boosting", "sensitivity"]),
    "beats_checklist": bool(compare.loc["gradient boosting", "events_caught"]
                            > compare.loc["checklist", "events_caught"]),
    "test_events": int(y_test.sum()),
    "is_best": False,  # final models not chosen yet
}
print(pd.DataFrame([row]).T)

SAVE_TO_DB = False   # needs the app DB (docker compose up -d app_db)
if SAVE_TO_DB:
    from db.metrics import save_metrics
    save_metrics(**row)